In [1]:
import pandas as pd
import numpy as np
from statistics import mean
from scipy.stats import invgamma

In [2]:
positions = ["GK", "DEF", "MID", "FWD"]

In [3]:
#Define Gibbs sampler
# X is the predictor variables df
# y is the target df

def gibbs_bayesian_lasso(X, y, num_iter, lambda_, a_sigma, b_sigma, 
                        init_beta, init_sigma2, init_tau2):
    
    n, p = X.shape

    # standardise X
    X_mean = X.mean()
    X_std  = X.std().replace(0, 1)   # replace 0 std to avoid division by zero
    X = (X - X_mean) / X_std

  
    #normalise y
    y_tilda = y - mean(y)
    
    # define parameter variables
    beta = [0]*p if init_beta is None else init_beta
    sigma2 = init_sigma2
    tau2 = [1]*p if init_tau2 is None else init_tau2
    beta_samples = np.zeros((num_iter, p))  


    # precompute matrix calculations
    # '@' is matrix multiplication in python
    # 't' means transpose in my definitions
    XtX = X.T @ X
    Xt_ytilda = X.T @ y_tilda


    # Define a min for tau to avoid zero division error and also needs not to be too close to 0
    eps = 1e-12


    # loop each iteration
    for i in range(num_iter):

        # ======================================================
        # 1. beta | tau2, sigma2, y_tilda
        # ======================================================

        # D^{-1} as pandas diagonal DataFrame
        Dinv = pd.DataFrame(
            np.diag(1.0 / np.clip(tau2, eps, None)), # this places a min on tau so cant be too close to 0
            index=X.columns, # just names the columns and rows 
            columns=X.columns
        )

        A = XtX + Dinv

        # define mean and covariance parameters for our Normal distribution
        beta_mean = np.linalg.solve(A, Xt_ytilda)
        covariance_matrix = sigma2 * np.linalg.inv(A)

        # define beta
        beta = np.random.multivariate_normal(
            mean=beta_mean.flatten(),     # make it 1D
            cov=covariance_matrix)



        

        # ======================================================
        # 2. tau2 | beta, sigma2
        # ======================================================

        # 1/τj has Inverse gausian distribution, with parameter mu and scale(/shape) below
        # 1/τj^2 | βj,σ2 ∼ Inv-Gaussian( sqrt((λ^2 * σ^2)/β^2), λ^2)

        for j in range(p):
            beta_j_squared = max(beta[j]**2, eps)
            mu_tau = np.sqrt((lambda_**2 * sigma2) / beta_j_squared)

            # note that wald distribution is inverse gaussian
            inv_tau2 = np.random.wald(mu_tau, lambda_**2)
            tau2[j] = 1.0 / max(inv_tau2, eps)


        

        # ======================================================
        # 3. sigma2 | beta, tau2, y
        # ======================================================

        # we place a non-zero prior on the two parameters of the inverse gamma dist of sigma squared, a_sigma and b_sigma
        # using small positive values gives a proper posterior as oppsed to using (0,0) and dont make a difference

        ### precomputing of values
        # residual = y_tilda - X beta
        residual = y_tilda - X @ beta
        # first quadratic form: (y - Xβ)^T (y - Xβ)
        term1 = residual @ residual
        # second quadratic form: β^T D^{-1} β - this is still correct and quicker
        term2 = np.sum((beta**2) / np.clip(tau2, eps, None))

        
        shape = a_sigma + (((n - 1) + p)/2)
        scale = b_sigma + ((term1 + term2)/2)

        sigma2 = invgamma.rvs(a=shape, scale=scale)


        # ======================================================
        # 4. add current iteration beta sample, beta[i], to total list
        # ======================================================
    
        beta_samples[i] = beta


    return beta_samples, X_mean, X_std 

In [4]:
def optimise_lambda(X_train, y_train, X_test, y_test, lambda_grid):
    best_lambda = None
    best_r2 = -np.inf

    for lambda_ in lambda_grid:
        beta_samples, X_mean, X_std = gibbs_bayesian_lasso(   # <-- unpack
            X_train, y_train, num_iter=1000, lambda_=lambda_,
            a_sigma=0.01, b_sigma=0.01, init_beta=None, init_sigma2=1, init_tau2=None)

        burn_in = 250
        beta_mean = beta_samples[burn_in:].mean(axis=0)

        y_mean = np.mean(y_train)
        X_test_scaled = (X_test - X_mean) / X_std   # <-- apply same scaling to test
        predictions = X_test_scaled @ beta_mean + y_mean
        residuals = y_test - predictions

        SS_res = np.sum(residuals ** 2)
        SS_tot = np.sum((y_test - np.mean(y_test)) ** 2)
        r2 = 1 - SS_res / SS_tot

        if r2 > best_r2:
            best_r2 = r2
            best_lambda = lambda_

    print(f"Optimal lambda for {pos} is: {best_lambda} (R² = {best_r2:.4f})")
    return best_lambda


In [5]:
# Combine data function
def combine_position_data(pos, start_week, end_week):
    combined_data = []

    base_path = "/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Individual GW+position files"

    for i in range(start_week, end_week + 1):
        file_name = f"{pos}_GW_{i}_players.csv"
        file_path = f"{base_path}/{file_name}"

        try:
            data = pd.read_csv(file_path)
            data["gameweek"] = i
            combined_data.append(data)
        except FileNotFoundError:
            print(f"File not found for: {file_name}")

    if combined_data:
        combined_data = pd.concat(combined_data, ignore_index=True)
        combined_data = combined_data.dropna()
    else:
        combined_data = pd.DataFrame()

    return combined_data


# now we move onto training
- now that we have define some generic functions as above, we can move onto our training data
- first we find the optimal lambda for each position

In [6]:
# Store optimal lambda for each position
optimal_lambdas = {}

for pos in positions:
    print(f"Working on pos {pos}")

    train_data = combine_position_data(pos, 1, 114)    # seasons 21/22/23
    test_data  = combine_position_data(pos, 115, 152)  # season 24

    y_train = train_data["total_points"]
    y_test  = test_data["total_points"]

    rolling_columns = [
        "team", "opponent_team", "value", "was_home",
        "ewma_total_points", "ewma_influence", "ewma_creativity", "ewma_threat",
        "avg_total_points_20", "avg_influence_20", "avg_creativity_20", "avg_threat_20"
    ]

    X_train = train_data[rolling_columns]
    X_test  = test_data[rolling_columns]

    lambda_grid = np.arange(0.01, 2.51, 0.25)
    optimal_lambda = optimise_lambda(X_train, y_train, X_test, y_test, lambda_grid)

    optimal_lambdas[pos] = optimal_lambda



Working on pos GK
File not found for: GK_GW_75_players.csv
File not found for: GK_GW_83_players.csv
Optimal lambda for GK is: 2.26 (R² = 0.2431)
Working on pos DEF
File not found for: DEF_GW_83_players.csv
Optimal lambda for DEF is: 1.01 (R² = 0.1603)
Working on pos MID
File not found for: MID_GW_83_players.csv
Optimal lambda for MID is: 1.01 (R² = 0.2501)
Working on pos FWD
File not found for: FWD_GW_83_players.csv
Optimal lambda for FWD is: 2.26 (R² = 0.2146)


In [7]:
###########################
# PERFORMANCE TESTING & PLOTTING (with 3 chains)

all_samples = pd.DataFrame(
    columns=rolling_columns + ["Chain", "Position"]
)

weekly_metrics_list = {}

X_means = {}
X_stds  = {}

beta_means = []

for pos in positions:
    print(f"Evaluating prediction performance for: {pos}")

    train_data = combine_position_data(pos, 1, 152)

    y_train = train_data["total_points"]

    predictor_variables = [
        "team", "opponent_team", "value", "was_home",
        "ewma_total_points", "ewma_influence", "ewma_creativity", "ewma_threat",
        "avg_total_points_20", "avg_influence_20",
        "avg_creativity_20", "avg_threat_20"
    ]

    X_train = train_data[rolling_columns]

    # --- new lines ---
    X_means[pos] = X_train.mean()
    X_stds[pos]  = X_train.std().replace(0, 1)
    # -----------------

    n, p = X_train.shape

    optimal_lambda = optimal_lambdas[pos]

    # Chain 1: default
    chain1, _, _  = gibbs_bayesian_lasso(
        X_train, y_train,
        lambda_=optimal_lambda,
        init_beta=np.zeros(p),
        init_sigma2=1.0,
        init_tau2=np.ones(p), num_iter=1000, a_sigma=0.01, b_sigma=0.01
    )
    chain1_df = pd.DataFrame(chain1, columns=rolling_columns)
    chain1_df["Chain"] = "Chain 1"

    # Chain 2: random start
    chain2, _, _  = gibbs_bayesian_lasso(
        X_train, y_train,
        lambda_=optimal_lambda,
        init_beta=np.random.uniform(-5, 5, p),
        init_sigma2=2.0,
        init_tau2=np.random.uniform(0.5, 2, p), num_iter=1000, a_sigma=0.01, b_sigma=0.01
    )
    chain2_df = pd.DataFrame(chain2, columns=rolling_columns)
    chain2_df["Chain"] = "Chain 2"

    # Chain 3: large start
    chain3, _, _  = gibbs_bayesian_lasso(
        X_train, y_train,
        lambda_=optimal_lambda,
        init_beta=np.full(p, 10.0),
        init_sigma2=5.0,
        init_tau2=np.full(p, 5.0), num_iter=1000, a_sigma=0.01, b_sigma=0.01
    )
    chain3_df = pd.DataFrame(chain3, columns=rolling_columns)
    chain3_df["Chain"] = "Chain 3"

    burn_in = 250

    combined_samples = pd.concat([
        chain1_df.loc[burn_in:, predictor_variables],
        chain2_df.loc[burn_in:, predictor_variables],
        chain3_df.loc[burn_in:, predictor_variables]
    ])

    beta_mean = combined_samples.mean(axis=0)
    beta_means.append((beta_mean, pos))


Evaluating prediction performance for: GK
File not found for: GK_GW_75_players.csv
File not found for: GK_GW_83_players.csv
Evaluating prediction performance for: DEF
File not found for: DEF_GW_83_players.csv
Evaluating prediction performance for: MID
File not found for: MID_GW_83_players.csv
Evaluating prediction performance for: FWD
File not found for: FWD_GW_83_players.csv


In [8]:
beta_means

[(team                   0.005553
  opponent_team          0.149326
  value                  0.294198
  was_home               0.028569
  ewma_total_points      0.436995
  ewma_influence         0.642476
  ewma_creativity        0.015660
  ewma_threat            0.043469
  avg_total_points_20    0.258352
  avg_influence_20      -0.024174
  avg_creativity_20      0.045429
  avg_threat_20         -0.048068
  dtype: float64,
  'GK'),
 (team                  -0.015616
  opponent_team          0.246949
  value                  0.156542
  was_home               0.104025
  ewma_total_points      0.233231
  ewma_influence         0.544770
  ewma_creativity        0.183055
  ewma_threat            0.045123
  avg_total_points_20    0.147090
  avg_influence_20      -0.078174
  avg_creativity_20     -0.021038
  avg_threat_20          0.028458
  dtype: float64,
  'DEF'),
 (team                   0.025954
  opponent_team          0.122072
  value                  0.125903
  was_home               0.